In [1]:
import pickle
import sys
import copy
import time

import cobra
import sympy
import pandas as pd
import numpy as np

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

No objective coefficients in model. Unclear what should be optimized


In [2]:
from macromolecules.RNA import RNA, mRNA
from macromolecules.protein import Protein
from macromolecules.macromolecule import Macromolecule
from macromolecules.protein import Protein

In [3]:
lp_path = '/data2/hratch/human_me/test_lp/'
with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
    me_model = pickle.load(handle)

In [ ]:
# not Metabolites, not proxy (on its own), not mRNA on its own
# HGNC yes


In [ ]:
lp_path = '/data2/hratch/human_me/test_lp/'

def add_metabolite(am = [], mu_val = 1e-9):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    ra = []
    for m in am: #am:
        r = cobra.Reaction('TEST_' + m.id)
        r.add_metabolites({m: 1})
        ra.append(r)
    
    me_model.add_reactions(ra)
    sln, status, _ = me_model.solve_lp(mu_val = mu_val)
    return sln, status



In [73]:
scale_proxy = 1/1e2
scale_mrna = None
with open(lp_path + 'toy_me_model_2.pickle', 'rb') as handle:
    me_model = pickle.load(handle)

new_reactions=[]

for r in me_model.reactions:
    if ('TRANSLATION_ELONGATION' in r.id or '_co_TRANSLOC_IMPORTtr' in r.id) and r.id != 'TRANSLATION_ELONGATIONc_COMPLEX_FORMATIONc':
        rxn = {k:v for k,v in r.metabolites.items() if '_mrna[c]' not in k.id and 'mrna_deg_proxy' not in k.id}

        to_add = set(r.metabolites.keys()).difference(rxn.keys())
        for m in to_add:
            if scale_mrna is not None:
                if '_mrna[c]' in m.id:
                    rxn[m] = r.metabolites[m]*scale_mrna
            if 'mrna_deg_proxy' in m.id:
                rxn[m] = r.metabolites[m]*scale_proxy

        r_ = func.ME_Reaction(r.id, type_ = r.type)
        r_.lower_bound = r.lower_bound
        r_.upper_bound = r.upper_bound
        r_.gene_reaction_rule = r.gene_reaction_rule
        r_.add_metabolites(rxn)

        new_reactions.append(r_)
    else: 
        new_reactions.append(r.copy())

me_model = func.ME_Model('')
me_model.add_reactions(new_reactions)
sln, status, _ = me_model.solve_lp(mu_val = 1e-9) 

Getting MINOS parameters...
Done in 244.003 seconds with status 0


In [17]:
# res = pd.DataFrame(columns = ['mrna_coef', 'proxy_coef', 'status'])
# counter = 0

# replace_mrna = 0
# for replace_proxy in tqdm([1,10]):

#     with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
#         me_model = pickle.load(handle)

#     new_reactions=[]

#     for r in me_model.reactions:
#         if ('TRANSLATION_ELONGATION' in r.id or '_co_TRANSLOC_IMPORTtr' in r.id) and r.id != 'TRANSLATION_ELONGATIONc_COMPLEX_FORMATIONc':
#             rxn = {k:v for k,v in r.metabolites.items() if '_mrna[c]' not in k.id and 'mrna_deg_proxy' not in k.id}

#             to_add = set(r.metabolites.keys()).difference(rxn.keys())
#             for m in to_add:
#                 if replace_mrna is not None:
#                     if '_mrna[c]' in m.id:
#                         rxn[m] = -replace_mrna
#                     else:
#                         rxn[m] = -replace_proxy

#             r_ = func.ME_Reaction(r.id, type_ = r.type)
#             r_.lower_bound = r.lower_bound
#             r_.upper_bound = r.upper_bound
#             r_.gene_reaction_rule = r.gene_reaction_rule
#             r_.add_metabolites(rxn)

#             new_reactions.append(r_)
#         else: 
#             new_reactions.append(r.copy())

#     me_model = func.ME_Model('')
#     me_model.add_reactions(new_reactions)
#     sln, status, _ = me_model.solve_lp(mu_val = 1e-9) 
    
#     res.loc[counter, :] = [replace_mrna, replace_proxy, status.max()]
#     counter +=1

In [27]:
# lp_path = '/data2/hratch/human_me/test_lp/'

# # error_metabolites = ['pre40s_rrna_protein_COMPLEX_FORMATIONn_protein_complex[n]']
# # precurors_work = ['HGNC:21173_folded_protein[n]', 'HGNC:32790_folded_protein[n]']
# # precursors_fail = ['HGNC:25542_folded_protein[n]', 'HGNC:29100_folded_protein[n]']

# # error_metabolites = precursors_fail.copy()

# def remove_metabolite(test_metabolites = [], mu_val = 0.01):
    
#     with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
#         me_model = pickle.load(handle)
    
#     me_metabolites = [m.id for m in me_model.metabolites if 'deg_proxy' in m.id or ('mrna[n]' in m.id and 'premrna' not in m.id and 'lariats' not in m.id)]
#     me_metabolites = []
#     me_metabolites += error_metabolites

#     for tm in test_metabolites:
#         me_metabolites.remove(tm)
    
#     ra = []
#     for mm_id in me_metabolites: #me_metabolites:
#         try:
#             mm_obj = me_model.metabolites.get_by_id(mm_id)
#         except:
#             mm_obj = params.human_model.metabolites.get_by_id(mm_id)
#         r = cobra.Reaction('TEST_' + mm_obj.id)
#         r.add_metabolites({mm_obj: 1}, reversibly = True)
#         ra.append(r)
#     if len(ra) > 0:
#         me_model.add_reactions(ra)
#     sln, status, _ = me_model.solve_lp(mu_val = mu_val)
#     return sln, status

# #max 1e-4
# sln,status = remove_metabolite(mu_val = 0)